In [ ]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc
from dash import html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64

# Configure OS routines
import os

# Required in Codio so the Dash server routes through the Jupyter proxy
JupyterDash.infer_jupyter_proxy_config()

# Configure the plotting routines
import numpy as np
import pandas as pd

# CRUD Python module developed in Project One
from CRUD_Python_Module import AnimalShelter

###########################
# Data Manipulation / Model
###########################

# Credentials for the aacuser account.
# Replaced with a placeholder for this public portfolio copy.
username = "aacuser"
password = "SET_YOUR_PASSWORD"

# Connect to the database through the CRUD module
db = AnimalShelter(username, password)

# An empty query document returns every record in the collection
df = pd.DataFrame.from_records(db.read({}))

# MongoDB returns an ObjectId '_id' column that the DataTable cannot render, so drop it
if '_id' in df.columns:
    df.drop(columns=['_id'], inplace=True)


#####################################
# Rescue Filter Queries (Controller)
#####################################
# Each function builds a MongoDB query document and sends it through the CRUD
# module read() method, so all filtering happens in the database, not in pandas.

def query_water_rescue():
    """Intact Female dogs, 26 to 156 weeks, breeds suited to water rescue."""
    return {
        "animal_type": "Dog",
        "breed": {"$in": ["Labrador Retriever Mix",
                          "Chesapeake Bay Retriever",
                          "Newfoundland"]},
        "sex_upon_outcome": "Intact Female",
        "age_upon_outcome_in_weeks": {"$gte": 26.0, "$lte": 156.0}
    }


def query_mountain_rescue():
    """Intact Male dogs, 26 to 156 weeks, breeds suited to mountain rescue."""
    return {
        "animal_type": "Dog",
        "breed": {"$in": ["German Shepherd",
                          "Alaskan Malamute",
                          "Old English Sheepdog",
                          "Siberian Husky",
                          "Rottweiler"]},
        "sex_upon_outcome": "Intact Male",
        "age_upon_outcome_in_weeks": {"$gte": 26.0, "$lte": 156.0}
    }


def query_disaster_rescue():
    """Intact Male dogs, 20 to 300 weeks, breeds suited to disaster or tracking work."""
    return {
        "animal_type": "Dog",
        "breed": {"$in": ["Doberman Pinscher",
                          "German Shepherd",
                          "Golden Retriever",
                          "Bloodhound",
                          "Rottweiler"]},
        "sex_upon_outcome": "Intact Male",
        "age_upon_outcome_in_weeks": {"$gte": 20.0, "$lte": 300.0}
    }


# Maps each radio button value to the matching query document
RESCUE_QUERIES = {
    'water': query_water_rescue,
    'mountain': query_mountain_rescue,
    'disaster': query_disaster_rescue
}


def get_filtered_dataframe(filter_type):
    """Run the selected filter through the CRUD module and return a clean DataFrame."""
    if filter_type in RESCUE_QUERIES:
        query = RESCUE_QUERIES[filter_type]()
    else:
        query = {}  # Reset returns the unfiltered data set

    records = db.read(query)
    frame = pd.DataFrame.from_records(records)

    if frame.empty:
        # Preserve the column structure so the table and charts stay stable
        return pd.DataFrame(columns=df.columns)

    if '_id' in frame.columns:
        frame.drop(columns=['_id'], inplace=True)
    return frame


#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

# Grazioso Salvare logo, encoded for inline display
image_filename = 'Grazioso Salvare Logo.png'
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

app.layout = html.Div([
    # Branding: client logo, dashboard title, and unique identifier
    html.Center([
        html.A([
            html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode()),
                     style={'height': '200px', 'width': 'auto'})
        ], href='https://www.snhu.edu', target='_blank'),
        html.B(html.H1('SNHU CS-340 Dashboard')),
        html.H4('Grazioso Salvare Search and Rescue Candidate Finder'),
        html.B(html.H5('Created by Luca Formento'))
    ]),
    html.Hr(),

    # Interactive filtering options
    html.Div([
        html.B('Filter by Rescue Type:'),
        dcc.RadioItems(
            id='filter-type',
            options=[
                {'label': 'Water Rescue', 'value': 'water'},
                {'label': 'Mountain or Wilderness Rescue', 'value': 'mountain'},
                {'label': 'Disaster or Individual Tracking', 'value': 'disaster'},
                {'label': 'Reset (All Animals)', 'value': 'reset'}
            ],
            value='reset',
            labelStyle={'display': 'inline-block', 'margin-right': '25px'},
            inputStyle={'margin-right': '6px'}
        )
    ], style={'text-align': 'center', 'font-size': '18px', 'padding': '10px'}),
    html.Hr(),

    # Interactive data table
    dash_table.DataTable(
        id='datatable-id',
        columns=[{"name": i, "id": i, "deletable": False, "selectable": True}
                 for i in df.columns],
        data=df.to_dict('records'),
        editable=False,
        filter_action="native",
        sort_action="native",
        sort_mode="multi",
        column_selectable="single",
        row_selectable="single",
        selected_columns=[],
        selected_rows=[0],
        page_action="native",
        page_current=0,
        page_size=10,
        style_table={'overflowX': 'auto'},
        style_cell={'textAlign': 'left', 'minWidth': '120px'},
        style_header={'backgroundColor': '#D2F3FF', 'fontWeight': 'bold'}
    ),
    html.Br(),
    html.Hr(),

    # Charts placed side by side
    html.Div(className='row',
             style={'display': 'flex'},
             children=[
                 html.Div(id='graph-id', className='col s12 m6'),
                 html.Div(id='map-id', className='col s12 m6')
             ])
])

#############################################
# Interaction Between Components / Controller
#############################################


@app.callback([Output('datatable-id', 'data'),
               Output('datatable-id', 'columns'),
               Output('datatable-id', 'selected_rows')],
              [Input('filter-type', 'value')])
def update_dashboard(filter_type):
    """Rebuild the data table from a MongoDB query whenever the filter changes."""
    dff = get_filtered_dataframe(filter_type)
    columns = [{"name": i, "id": i, "deletable": False, "selectable": True}
               for i in dff.columns]
    data = dff.to_dict('records')
    selected = [0] if len(data) > 0 else []
    return data, columns, selected


@app.callback(Output('graph-id', "children"),
              [Input('datatable-id', "derived_virtual_data")])
def update_graphs(viewData):
    """Pie chart of the breed distribution currently shown in the data table."""
    if viewData is None:
        return []

    dff = pd.DataFrame.from_dict(viewData)
    if dff.empty or 'breed' not in dff.columns:
        return [html.H5('No records match the selected filter.')]

    # Group the long tail of breeds so the chart stays readable
    counts = dff['breed'].value_counts().reset_index()
    counts.columns = ['breed', 'count']
    if len(counts) > 10:
        top = counts.head(10)
        other = pd.DataFrame([['Other Breeds', counts['count'][10:].sum()]],
                             columns=['breed', 'count'])
        counts = pd.concat([top, other], ignore_index=True)

    return [
        dcc.Graph(
            figure=px.pie(counts, names='breed', values='count',
                          title='Breed Distribution of Filtered Animals')
        )
    ]


@app.callback(Output('datatable-id', 'style_data_conditional'),
              [Input('datatable-id', 'selected_columns')])
def update_styles(selected_columns):
    """Highlight the column the user selects."""
    if selected_columns is None:
        return []
    return [{
        'if': {'column_id': i},
        'background_color': '#D2F3FF'
    } for i in selected_columns]


@app.callback(Output('map-id', "children"),
              [Input('datatable-id', "derived_virtual_data"),
               Input('datatable-id', "derived_virtual_selected_rows")])
def update_map(viewData, index):
    """Plot the selected animal on a map of the Austin, TX region."""
    if viewData is None:
        return
    dff = pd.DataFrame.from_dict(viewData)
    if dff.empty:
        return [html.H5('No location to display for the selected filter.')]

    # Only single row selection is enabled, so the list holds one index
    if index is None or len(index) == 0:
        row = 0
    else:
        row = index[0]

    lat = dff.iloc[row]['location_lat']
    lon = dff.iloc[row]['location_long']

    # Center on the selected animal so the marker is always in view
    return [
        dl.Map(style={'width': '100%', 'height': '500px'},
               center=[lat, lon], zoom=10, children=[
            dl.TileLayer(id="base-layer-id"),
            dl.Marker(position=[lat, lon], children=[
                dl.Tooltip(dff.iloc[row]['breed']),
                dl.Popup([
                    html.H1("Animal Name"),
                    html.P(dff.iloc[row]['name'])
                ])
            ])
        ])
    ]


app.run_server(debug=True, mode='inline', height=1200)
